# Homework: a decision diary, and one chart of your own, worked solution

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "messy" / "plays_messy.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)

sys.path.insert(0, "sessions/session-09/solutions")
from cleaning import clean_plays, load_messy

MESSY = "data/messy/plays_messy.csv"

raw = load_messy(MESSY)
plays = clean_plays(raw)
clean = pd.read_csv("data/clean/plays.csv")
print(len(raw), "rows messy,", len(plays), "rows cleaned")

## Task 1: which device gets skipped most?

In [ ]:
def skip_rates(df):
    """Return one row per device: number of plays, and percentage skipped, highest first."""
    rates = df.groupby("device")["skipped"].agg(plays="size", rate="mean")
    rates["rate"] = rates["rate"] * 100
    return rates.sort_values("rate", ascending=False)


def winner(rates):
    """Return a one-line summary of the top row of a skip_rates table."""
    device = rates.index[0]
    return f"{device!r}: {rates['rate'].iloc[0]:.1f}% of {rates['plays'].iloc[0]} plays"

`{device!r}` prints the value with quotes round it, the way `repr()`
does. When the values themselves are the problem, like `"CAR"` versus
`"car"` or `"phone "` with a space, quotes are the only way to see it.

### Step 1: as loaded

In [ ]:
step1 = skip_rates(raw)
print(winner(step1))
step1.head()

`"CAR"` in capitals, at 17.2%, from 29 plays. Every spelling of every
device is its own group, so the "winner" is just whichever small group
happened to have a few skips in it. Further down, `"tablet"` with 6 plays
is at 16.7%.

### Step 2: device text cleaned, nothing else

In [ ]:
step2_data = raw.copy()
step2_data["device"] = step2_data["device"].str.strip().str.lower()
step2_data = step2_data[step2_data["device"] != ""]

step2 = skip_rates(step2_data)
print(winner(step2))
step2

Five groups, and speaker wins at 14.1%. **This is the step that changed the
answer.**

### Step 3: duplicates dropped too

In [ ]:
step3 = skip_rates(step2_data.drop_duplicates())
print(winner(step3))
step3

Still speaker, still 14.1%. The 40 copies were spread across devices, so
removing them moved the counts (1,024 phone plays became 1,004) but hardly
moved the rates. Dropping duplicates changes totals far more than it
changes percentages.

### Step 4: the full `clean_plays`

In [ ]:
step4 = skip_rates(plays)
print(winner(step4))
step4

Now `"unknown"` wins, at 15.9% of 69 plays.

### Step 5: the clean file

In [ ]:
step5 = skip_rates(clean)
print(winner(step5))
step5

In [ ]:
assert step1.index[0] == "CAR"
assert step2.index[0] == step3.index[0] == step5.index[0] == "speaker"
assert step4.index[0] == "unknown"
assert round(step4.loc["speaker", "rate"], 1) == 14.1
assert round(step5.loc["speaker", "rate"], 1) == 14.0

### The diary

| Step | Most skipped | Rate | Plays |
|---|---|---|---|
| 1. as loaded | `CAR` | 17.2% | 29 |
| 2. text cleaned | speaker | 14.1% | 333 |
| 3. duplicates dropped | speaker | 14.1% | 326 |
| 4. full clean, with "unknown" | unknown | 15.9% | 69 |
| 5. the clean file | speaker | 14.0% | 336 |

**1. Which decision changed the answer most?** Cleaning the text. Before
it, the question was not really being asked: you were ranking twenty
spellings, not five devices. After it, every later step picked the same
winner as the clean file, within 0.1 of a percentage point, as long as
`"unknown"` stays out of the ranking.

**2. What to do about `"unknown"`.** Keep it in the table, and keep it out
of the ranking. It is not a device; it is 69 plays on some mixture of
devices. "The most skipped device is unknown" is a sentence with no
meaning. The honest report is: "speaker is skipped most, at 14.1%; 69
plays (3%) had no device recorded." Labelling the gap kept it visible,
which is the point of labelling it. It did not turn it into a category you
can compare.

**3. Would `min_plays` have saved you?** The session 4 stretch only ranked
groups with enough plays to mean something. With a minimum of 30 plays,
`"CAR"` (29) and `"tablet"` (6) drop out of step 1. The new winner is
`"LAPTOP"`, in capitals, at 16.2% of 74 plays. Also wrong: in the clean file,
laptop's rate is 11.5%.

A minimum protects you from small groups. It cannot protect you from
asking the question about spellings instead of devices. Only cleaning does
that.

In [ ]:
step1_big = step1[step1["plays"] >= 30]
print(winner(step1_big))
assert step1_big.index[0] == "LAPTOP"

## Task 2: one chart that answers one question

**My question: which artists got the most minutes in 2025?**

In [ ]:
in_2025 = plays[plays["played_at"].str.startswith("2025")]
top_artists = (in_2025.groupby("artist_name")["minutes_played"].sum()
               .sort_values(ascending=False).head(8))
top_artists

DJ Kompas 538.8, Glass Tram 535.3. A gap of 3.5 minutes. Before writing
"DJ Kompas got the most minutes" in a title, check how much we do not know
about those two.

In [ ]:
missing_2025 = in_2025[in_2025["minutes_played"].isna()]
missing_2025["artist_name"].value_counts().head(4)

Both artists have plays in 2025 with no minutes recorded. A single one of
those could be worth more than 3.5 minutes, so the data cannot say who
won. The title should say what the data can support.

In [ ]:
os.makedirs("output", exist_ok=True)

ranked = top_artists.sort_values()                # barh draws from the bottom up

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(ranked.index, ranked.values, color="#4f6ddb")
ax.set_title("DJ Kompas and Glass Tram were neck and neck in 2025, "
             f"{top_artists.iloc[0] - top_artists.iloc[1]:.1f} minutes apart")
ax.set_xlabel("minutes played in 2025")
ax.bar_label(bars, fmt="{:,.1f}", padding=3)
ax.spines[["top", "right"]].set_visible(False)

fig.savefig("output/top_artists_2025.png", dpi=150, bbox_inches="tight")
plt.show()

**What I would say about it:** "Two artists dominated my 2025, DJ Kompas
and Glass Tram, with over 535 minutes each, and the gap between them is
smaller than the minutes we failed to record."

**What this does not show:** who actually won. DJ Kompas has 6 plays in
2025 with no minutes and Glass Tram has 2, and any one of them could be
longer than the 3.5-minute gap. It also says nothing about 2024, or about
how often each artist was played: this is minutes, not plays.

Choices worth explaining:

* **`barh`, horizontal bars**, because artist names are long and would
  overlap on a vertical axis. Still bars, because artists are categories.
* **sorted ascending before `barh`**, because horizontal bars are drawn
  from the bottom, so the biggest ends up on top
* **the top eight**, not all 38. The question is about the top; the tail
  would be noise

And the payoff of being careful: in the clean file, the one we would never
have in real life, the gap is much bigger.

In [ ]:
clean_2025 = clean[clean["played_at"].str.startswith("2025")]
clean_top = clean_2025.groupby("artist_name")["minutes_played"].sum().sort_values(ascending=False)
print(clean_top.head(2).round(1))

assert list(top_artists.index[:2]) == ["DJ Kompas", "Glass Tram"]
assert round(top_artists.iloc[0] - top_artists.iloc[1], 1) == 3.5
assert list(missing_2025["artist_name"].value_counts()[["DJ Kompas", "Glass Tram"]]) == [6, 2]

570.9 against 545.5: DJ Kompas did win, by 25.4 minutes. A title claiming
the win would have been right, by luck. "Neck and neck" was the honest
reading of the data we had. Being right by luck and being right because
you checked look identical on the day, and only one of them keeps working.

### A second example: a chart whose answer is "no"

**Do I listen more on some days of the week than others?**

In [ ]:
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
per_day = pd.to_datetime(plays["played_at"]).dt.day_name().value_counts().reindex(days)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(per_day.index, per_day.values, color="#4f6ddb")
ax.set_title("No favourite day: between 302 and 333 plays on every day of the week")
ax.set_ylabel("plays, 2024 and 2025")
ax.bar_label(bars)
ax.spines[["top", "right"]].set_visible(False)

fig.savefig("output/plays_per_weekday.png", dpi=150, bbox_inches="tight")
plt.show()

assert per_day.min() == 302 and per_day.max() == 333

"Nothing interesting here" is an answer, and a chart that shows it clearly
is a good chart.

**What this does not show:** minutes. It counts plays, so a day with the
same number of plays could still hold more listening time. And it adds 2024
and 2025 together, so a change between the two years would be hidden.

Two details:

* `.reindex(days)` puts the days in calendar order. Days are categories,
  so bars, but they do have a natural order, and alphabetical would scatter
  them.
* bars from zero make the "all the same" visible. Zoom the axis to start at
  300 and Friday suddenly looks like a party.